### Introduction to BERT and GPT
* What is BERT
* BERT (Bidirectional Encoder Representations From Transformers)
    * Developed by Google AI
    * Process input sequences bidirectionally, enabling it to capture context from both directions 
    * Pre-Trained on tasks like masked manguage modeling and next sentence predictions (NSP)

* Key Featuers
    * Bidirectional : understands context from both left and right sies of a word
    * Transformer Encode - Based: Optimized for understanding input text
    * Applications: Sentiment analysis

* What is GPT
    * Developed by OpenAI
    * Produces input sequences unidirectionally (left to right) , focusing on generative tasks
* Key  features
    * Unidirectional: Process text from left- ot right , focusing on text generation
    * Transformer Decoder - Based : Optimized for generating coherent text
    * Applications: text generation, chat bot
    

### Fine-Tuning Pre-Trained Models for Downstream tasks
---
* Why Fine Tuning
    * Pre-trained models are trained on large generic datasets
    * Fine tuning adapts them to specific tasks like sentipment analysis or classification
    * Steps to Fine - Tune
        * Load a Pre- Trained Model
            * Use Libraries like hugging face to load a pre-trained BERT or GPT model
        * Prepare Datasets
            * Format DAtaset for the specific task 
        * Train and Evaluate
            * Fine-tune the model using task-specific data
            

In [1]:
# !pip uninstall -y torch torchvision torchaudio
!pip install torch 

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import sys
from datasets import config

sys.modules.pop("torchvision", None)
config.TORCHVISION_AVAILABLE = False


In [3]:
# Load dataset
dataset = load_dataset("stanfordnlp/imdb")
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# Tokenize the dataset

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
# Prepare data for training
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label","labels")
tokenized_datasets.set_format("torch")

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

In [5]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 2 )

# training args
training_args = TrainingArguments(
    output_dir = "./",
    eval_strategy = "epoch",
     learning_rate = 2e-5,
    per_device_train_batch_size =8 ,
    per_device_eval_batch_size =8,
    num_train_epochs=3,
    weight_decay = 0.01,
    logging_dir = "./log",
    logging_steps=10,
    save_steps=500,
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `loggin

In [6]:
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,eval_dataset=test_dataset, processing_class=tokenizer)

In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.177133,0.251470
2,0.190384,0.257235
3,0.008067,0.307943


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=0.16633822521890204, metrics={'train_runtime': 10015.868, 'train_samples_per_second': 7.488, 'train_steps_per_second': 0.936, 'total_flos': 1.9733329152e+16, 'train_loss': 0.16633822521890204, 'epoch': 3.0})

In [11]:

print(trainer.create_model_card())


None


In [13]:
print(trainer.__hash__())
trainer.save_model(output_dir="./")

8478415111332


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
!ls -lha


total 419M
drwxr-xr-x 1 root root 4.0K Jun 17 10:13 .
drwxr-xr-x 1 root root 4.0K Jun 17 06:33 ..
drwxr-xr-x 2 root root 4.0K Jun 17 06:56 checkpoint-1000
drwxr-xr-x 2 root root 4.0K Jun 17 07:03 checkpoint-1500
drwxr-xr-x 2 root root 4.0K Jun 17 07:10 checkpoint-2000
drwxr-xr-x 2 root root 4.0K Jun 17 07:17 checkpoint-2500
drwxr-xr-x 2 root root 4.0K Jun 17 07:23 checkpoint-3000
drwxr-xr-x 2 root root 4.0K Jun 17 07:43 checkpoint-3500
drwxr-xr-x 2 root root 4.0K Jun 17 07:49 checkpoint-4000
drwxr-xr-x 2 root root 4.0K Jun 17 07:56 checkpoint-4500
drwxr-xr-x 2 root root 4.0K Jun 17 06:50 checkpoint-500
drwxr-xr-x 2 root root 4.0K Jun 17 08:03 checkpoint-5000
drwxr-xr-x 2 root root 4.0K Jun 17 08:10 checkpoint-5500
drwxr-xr-x 2 root root 4.0K Jun 17 08:16 checkpoint-6000
drwxr-xr-x 2 root root 4.0K Jun 17 08:36 checkpoint-6500
drwxr-xr-x 2 root root 4.0K Jun 17 08:45 checkpoint-7000
drwxr-xr-x 2 root root 4.0K Jun 17 08:52 checkpoint-7500
drwxr-xr-x 2 root root 4.0K Jun 17 08:58 checkpo

In [47]:
!pip install -U "huggingface_hub<0.26"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 8.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.25.2 which is incompatible.
transformers 5.10.2 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.25.2 which is incompatible.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.2 which is incompatible.


In [1]:
import os
!pip uninstall -y hf_xet
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import login


login("<token>")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [2]:
from huggingface_hub import HfApi, create_repo

repo_id = "Achiket/bert-base-uncased-imdb"

create_repo(repo_id, exist_ok=True)

api = HfApi()

api.upload_folder(
    folder_path="/content",
    repo_id=repo_id,
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "tokenizer_config.json",
        "tokenizer.json",
        "vocab.txt",
        "special_tokens_map.json",
    ],
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:100: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Achiket/bert-base-uncased-imdb/commit/296c4d7eaa4349dd25446178e913f44fc64934fb', commit_message='Upload folder using huggingface_hub', commit_description='', oid='296c4d7eaa4349dd25446178e913f44fc64934fb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Achiket/bert-base-uncased-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='Achiket/bert-base-uncased-imdb'), pr_revision=None, pr_num=None)

In [3]:
!ls -lha

total 419M
drwxr-xr-x 1 root root 4.0K Jun 17 10:32 .
drwxr-xr-x 1 root root 4.0K Jun 17 06:33 ..
drwxr-xr-x 2 root root 4.0K Jun 17 10:32 bert_imdb_finetuned
drwxr-xr-x 2 root root 4.0K Jun 17 06:56 checkpoint-1000
drwxr-xr-x 2 root root 4.0K Jun 17 07:03 checkpoint-1500
drwxr-xr-x 2 root root 4.0K Jun 17 07:10 checkpoint-2000
drwxr-xr-x 2 root root 4.0K Jun 17 07:17 checkpoint-2500
drwxr-xr-x 2 root root 4.0K Jun 17 07:23 checkpoint-3000
drwxr-xr-x 2 root root 4.0K Jun 17 07:43 checkpoint-3500
drwxr-xr-x 2 root root 4.0K Jun 17 07:49 checkpoint-4000
drwxr-xr-x 2 root root 4.0K Jun 17 07:56 checkpoint-4500
drwxr-xr-x 2 root root 4.0K Jun 17 06:50 checkpoint-500
drwxr-xr-x 2 root root 4.0K Jun 17 08:03 checkpoint-5000
drwxr-xr-x 2 root root 4.0K Jun 17 08:10 checkpoint-5500
drwxr-xr-x 2 root root 4.0K Jun 17 08:16 checkpoint-6000
drwxr-xr-x 2 root root 4.0K Jun 17 08:36 checkpoint-6500
drwxr-xr-x 2 root root 4.0K Jun 17 08:45 checkpoint-7000
drwxr-xr-x 2 root root 4.0K Jun 17 08:52 che

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("/content")
tokenizer = AutoTokenizer.from_pretrained("/content")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [19]:
import torch

text = """James Gunn's Superman is a refreshing return to what the character has always represented: hope, kindness, and optimism. Rather than trying to make Superman darker or morally conflicted, the film embraces his compassion and belief in humanity. David Corenswet delivers a convincing performance, capturing both Superman's strength and Clark Kent's humility. The chemistry among the cast feels natural, and the action sequences are visually exciting without overshadowing the emotional core of the story.

However, the movie isn't without flaws. One of its biggest issues is that it tries to do too much at once. With multiple heroes, villains, and subplots competing for attention, some characters feel underdeveloped and don't receive enough screen time to leave a lasting impact. Certain emotional moments also lose their effectiveness because the pacing moves too quickly, giving little room for scenes to breathe.

The film's humor is another mixed aspect. While much of it lands well and helps differentiate the movie from previous Superman adaptations, a few jokes interrupt scenes that would have benefited from a more serious tone. At times, the tonal shifts between comedy and drama can feel slightly uneven.

Lex Luthor is an engaging antagonist, but his motivations could have been explored in greater depth. While he serves as a formidable obstacle, the screenplay occasionally relies on familiar comic book tropes instead of giving him a more nuanced psychological presence.

Visually, the movie is colorful and embraces the comic-book aesthetic, which is a welcome change from the darker superhero films of the past decade. Some CGI-heavy moments, though, look less polished than the practical and character-driven scenes, making the spectacle occasionally feel artificial.

Despite these criticisms, Superman succeeds where it matters most. It reminds audiences why Superman remains one of the greatest superheroes ever created—not because he is the strongest, but because he chooses to do good even when it's difficult. The film isn't perfect, but it lays a strong foundation for the future of the DC Universe and delivers an inspiring, heartfelt superhero adventure."""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=1).item()

print("Predicted Class:", prediction)
print("Label:", model.config.id2label[prediction])
print("count", len(model.config.id2label))

Predicted Class: 1
Label: LABEL_1
count 2
